# Submission 01 - HistGradientBoosting

This notebook trains the HistGradientBoosting model on the full training dataset and creates the first Kaggle submission.

The model will predict the probability of `Will_Buy_EV = Yes`, which is required for the ROC-AUC competition metric.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

## 2. Load the Data

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)
    
print('Sample submission shape:', sample_submission.shape)

Train shape: (668665, 15)
Test shape: (286571, 14)
Sample submission shape: (286571, 2)


## 3. Prepare Features and Target

In [3]:
TARGET = 'Will_Buy_EV'
ID_COLUMN = 'id'

X = train.drop(columns=[TARGET, ID_COLUMN])
y = train[TARGET].map({'No': 0, 'Yes': 1})
    
X_test = test.drop(columns=[ID_COLUMN])
    
print('Training features:', X.shape)
print('Test features:', X_test.shape)
print('Positive class rate:', y.mean())

Training features: (668665, 13)
Test features: (286571, 13)
Positive class rate: 0.17464500160768098


## 4. Build the Preprocessing Pipeline

In [4]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

Numeric features: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
Categorical features: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


C:\Users\aakif\AppData\Local\Temp\ipykernel_60\3748426724.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object']).columns.tolist()


## 5. Train HistGradientBoosting

In [5]:
model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=200,
    max_leaf_nodes=31,
    random_state=42
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

print('Training model on the full training dataset...')
pipeline.fit(X, y)
print('Training complete.')

Training model on the full training dataset...
Training complete.


## 6. Generate Test Predictions

In [6]:
test_predictions = pipeline.predict_proba(X_test)[:, 1]

print('Prediction count:', len(test_predictions))
print('Minimum probability:', test_predictions.min())
print('Maximum probability:', test_predictions.max())
print('Missing predictions:', np.isnan(test_predictions).sum())

Prediction count: 286571
Minimum probability: 5.7419643842961733e-05
Maximum probability: 0.9639696881693935
Missing predictions: 0


## 7. Create Submission File

In [7]:
submission = sample_submission.copy()
submission[TARGET] = test_predictions
    
submission_path = '../submissions/submission_01.csv'
submission.to_csv(submission_path, index=False)
    
print('Submission saved to:', submission_path)
print('Submission shape:', submission.shape)
print('Submission columns:', submission.columns.tolist())
    
display(submission.head())

Submission saved to: ../submissions/submission_01.csv
Submission shape: (286571, 2)
Submission columns: ['id', 'Will_Buy_EV']


,id,Will_Buy_EV
0,668665,0.007683
1,668666,0.021129
2,668667,0.006921
3,668668,0.003305
4,668669,0.016229


## 8. Verify Submission

In [8]:
assert submission.shape == sample_submission.shape
assert submission.columns.tolist() == sample_submission.columns.tolist()
assert submission[TARGET].notna().all()
assert submission[TARGET].between(0, 1).all()
    
print('Submission verification passed.')
    
print('\nFirst 5 predictions:')
print(submission.head().to_string(index=False))

Submission verification passed.

First 5 predictions:
    id  Will_Buy_EV
668665     0.007683
668666     0.021129
668667     0.006921
668668     0.003305
668669     0.016229
